<a href="https://colab.research.google.com/github/kandlaguntaramesh/prompt_engineering_homework/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q "google-auth==2.49.0"

In [4]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

print("API key loaded successfully")

API key loaded successfully


In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0
)

print("Gemini initialized successfully")

Gemini initialized successfully


In [20]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)



Saving california_housing_train.csv to california_housing_train (1).csv


In [22]:
# Randomly select 70 rows from the dataset

few_shot_df = df.sample(
    n=70,
    random_state=42
)

In [24]:
def create_few_shot_prompt(examples):

    prompt = """
You are a home price prediction assistant.

Your task is to predict the median house value
using California housing data.

Below are examples containing housing attributes
and their actual median house values.

Use these examples as few-shot examples to learn
the relationship between the house attributes and price.

FEW-SHOT EXAMPLES:
"""

    for _, row in examples.iterrows():

        prompt += f"""

Example:
longitude: {row['longitude']}
latitude: {row['latitude']}
housing_median_age: {row['housing_median_age']}
total_rooms: {row['total_rooms']}
total_bedrooms: {row['total_bedrooms']}
population: {row['population']}
households: {row['households']}
median_income: {row['median_income']}

Actual median_house_value: {row['median_house_value']}
"""

    return prompt


few_shot_prompt = create_few_shot_prompt(few_shot_df)

print(few_shot_prompt[:5000])


You are a home price prediction assistant.

Your task is to predict the median house value
using California housing data.

Below are examples containing housing attributes
and their actual median house values.

Use these examples as few-shot examples to learn
the relationship between the house attributes and price.

FEW-SHOT EXAMPLES:

        
Example:
longitude: -120.87
latitude: 37.77
housing_median_age: 9.0
total_rooms: 4838.0
total_bedrooms: 920.0
population: 2460.0
households: 923.0
median_income: 3.5959

Actual median_house_value: 142700.0

        
Example:
longitude: -118.14
latitude: 34.11
housing_median_age: 52.0
total_rooms: 2742.0
total_bedrooms: 422.0
population: 1153.0
households: 414.0
median_income: 8.1124

Actual median_house_value: 500001.0

        
Example:
longitude: -120.05
latitude: 36.98
housing_median_age: 16.0
total_rooms: 3705.0
total_bedrooms: 739.0
population: 2463.0
households: 697.0
median_income: 2.5288

Actual median_house_value: 61800.0

        
Exa

In [12]:
new_house = {
    "longitude": -122.23,
    "latitude": 37.88,
    "housing_median_age": 35,
    "total_rooms": 2500,
    "total_bedrooms": 500,
    "population": 900,
    "households": 450,
    "median_income": 5.5
}

In [25]:
new_house_prompt = f"""

NEW HOUSE TO PREDICT:

longitude: {new_house['longitude']}
latitude: {new_house['latitude']}
housing_median_age: {new_house['housing_median_age']}
total_rooms: {new_house['total_rooms']}
total_bedrooms: {new_house['total_bedrooms']}
population: {new_house['population']}
households: {new_house['households']}
median_income: {new_house['median_income']}

Based on the few-shot examples above, predict the
median house value for this new house.

Return only the predicted price as a number.
"""

final_prompt = few_shot_prompt + new_house_prompt


In [16]:
response = model.invoke(final_prompt)

price = float(response.content[0]["text"])

print("Predicted Home Price: ${:,.2f}".format(price))

Predicted Home Price: $350,000.00


In [19]:
print("Enter the new house information:")

longitude = float(input("Longitude: "))
latitude = float(input("Latitude: "))
housing_median_age = float(input("Housing median age: "))
total_rooms = float(input("Total rooms: "))
total_bedrooms = float(input("Total bedrooms: "))
population = float(input("Population: "))
households = float(input("Households: "))
median_income = float(input("Median income: "))

new_house_prompt = f"""

NEW HOUSE TO PREDICT:

longitude: {longitude}
latitude: {latitude}
housing_median_age: {housing_median_age}
total_rooms: {total_rooms}
total_bedrooms: {total_bedrooms}
population: {population}
households: {households}
median_income: {median_income}

Predict the median house value for this house.

Return only the predicted price as a number.
"""
final_prompt = few_shot_prompt + new_house_prompt

response = model.invoke(final_prompt)

price = float(response.content[0]["text"])

print("Predicted Home Price: ${:,.2f}".format(price))



Enter the new house information:
Longitude: -135.96
Latitude: 78.23
Housing median age: 19
Total rooms: 15
Total bedrooms: 9
Population: 4
Households: 2
Median income: 5699
Predicted Home Price: $500,001.00
